## Install Dependencies

In [ ]:
!pip install langchain langchain-groq

## Imports

In [ ]:
import os
from langchain.agents import create_agent
# Note: SummarizationMiddleware is a conceptual pattern for handling memory/tokens automatically
from langchain.agents.middleware import SummarizationMiddleware
from langchain.tools import Tool

## Define Tools

In [ ]:
def get_weather(location: str) -> str:
    return f"The weather in {location} is 72F and sunny."

weather_tool = Tool(name="Weather", func=get_weather, description="Get the weather")
calculator_tool = Tool(name="Calculator", func=lambda x: eval(x), description="Math operations")

## Encapsulated Middleware Agent Function

In [ ]:
def run_agent_with_middleware(query: str):
    # Set Groq API Key
    os.environ['GROQ_API_KEY'] = 'gsk_YOUR_API_KEY_HERE'

    # Create Agent with Summarization Middleware
    # This automatically summarizes old messages to save tokens!
    agent = create_agent(
        model="llama-3.3-70b-versatile",          # Main reasoning model (Groq)
        tools=[weather_tool, calculator_tool],
        middleware=[
            SummarizationMiddleware(
                model="llama3-8b-8192",           # Smaller model used just for summarizing background memory
                trigger=("tokens", 4000),         # Trigger summary when conversation hits 4000 tokens
                keep=("messages", 20),            # Always keep the most recent 20 messages untouched
            ),
        ],
    )
    
    return agent.invoke({"input": query})

## Test the Middleware

In [ ]:
answer = run_agent_with_middleware("What is the weather in New York, and what is 50 * 4?")